<a href="https://colab.research.google.com/github/ztoth8684/calcium_signaling/blob/main/Calcium_Frequency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This is the updated script for generating frequency data from 4x Lionheart images
#Current as of 1/19/2026



In [ ]:
!pip install cellpose tifffile scipy scikit-image joblib

In [ ]:
import numpy as np
import tifffile
from matplotlib import pyplot as plt
from cellpose import models
import torch
from scipy.signal import find_peaks
from skimage.measure import label, regionprops, regionprops_table
import math
from joblib import Parallel, delayed

#utility functions (use in other code segments as well)

#######################################################################################################################################################
#function to extract timeseries data from image stack. This one takes a separate segmentation image (use max z projection image)
#inputs:
    #stack: a 3D numpy array of the timeseries image
    #seg_img: an image to base segmentation off of. It can be a single slice of the stack, or it could be a max z projection image
    #diam: diameter of the cell the segmentation algorithm expects
#outputs:
    #data: an n-by-t 2D numpy array, containing avg calcium signal intensity for each n cell for t timepoints
    #mask: an x-by-y 2D numpy array with the same size as the seg_img input. Each value is either 0 for no cell, or [1...n] for each cell ID
    #morph: an n-by-1 1D numpy array containing the circularity value for each cell
    #size: an n-by-1 1D numpy array containing the size of the cell in pixels

def extract_data_wloc(stack, seg_img, diam=15):

    #segmenting the image to generate a mask of each cell
    img = seg_img
    model = models.CellposeModel(gpu=True, model_type = 'cyto2', device=torch.device('cuda'))
    mask, flows, styles = model.eval(img, diameter=diam, flow_threshold=0.2, channels=[0,0])

    #morphological characteristics
    regions = regionprops(mask)
    morph = np.zeros(mask.max()-1)
    size = np.zeros(mask.max()-1)

    for i in range(len(regions)-1):
        morph[i] = regions[i+1].axis_major_length/regions[i+1].axis_minor_length
        size[i] = mask[mask==i+1].shape[0]

    #calcium data extraction using multiprocessing. The n_jobs should be set to less than 60 for compatibility issues
    t_len = stack.shape[0]
    numcell = mask.max()-1

    def process(i, stack, mask):
        tdata=np.zeros(t_len)
        for t in range(t_len):
            a,b = np.where(mask==i+1)
            val=0
            img = stack[t,:,:]
            for j in range(a.shape[0]):
                val = val + img[a[j], b[j]]
            tdata[t] = val/a.shape[0]
        return tdata

    results = Parallel(n_jobs=59)(delayed(process)(i, stack, mask) for i in range(numcell))
    data = np.stack(results, axis=0)

    return data, mask, morph, size

#######################################################################################################################################################
#function to apply a polynomial correction to the timeseries data for each cell to correct for autofluorescence drift
#inputs:
    #data: an n-by-t 2D numpy array, containing avg calcium signal intensity for each n cell for t timepoints
    #n_poly: the order of the polynomial to fit the function to. Should be somewhere between 4-8
#outputs:
    #result: an n-by-t 2D numpy array, containing the input data with correction applied
def poly_corr(data, n_poly):
    result = np.zeros(data.shape)
    for cell in range(data.shape[0]):
        y = np.linspace(0,data.shape[1]-1,data.shape[1])
        model = np.poly1d(np.polyfit(y,data[cell,:],n_poly))
        poly = model(y)
        result[cell,:] = np.subtract(data[cell,:], poly)
    return result

#######################################################################################################################################################
#function to get timepoint on where the wave starts
#inputs:
    #x: an n-by-t 2D numpy array, containing avg calcium signal intensity for each n cell for t timepoints
    #thres: int value for the calcium intensity threshold where it will be considered the beginning of a "wave"
    #mask: an optional parameter, an x-by-y 2D numpy array with the same size as the img input. Each value is either 0 for no cell, or [1...n] for each cell ID
#outputs:
    #wave_begin: an n-by-1 1D numpy array with the timepoint at which the wave starts for each cell. It contains -1 for all cells that never reach the threshold value
def get_wave_start(x, thres, mask=None):

    wave_begin = np.zeros(x.shape[0])
    for i in range(x.shape[0]):
        try:
            wave_begin[i] = np.where(x[i,:]>thres)[0][0]
        except:
            wave_begin[i] = -1

    if mask is not None:
        new_mask = np.zeros(mask.shape)
        for i in range(mask.max()-1):
            new_mask = np.where(mask==i+1, wave_begin[i], new_mask)
        return wave_begin, new_mask
    else:
        return wave_begin

#######################################################################################################################################################
#function to select spiking cells
#inputs:
    #data: an n-by-t 2D numpy array, containing avg calcium signal intensity for each n cell for t timepoints
    #prom: int value for the prominence threshold for the spike
    #pkct_min: an optional parameter, min number of peaks to consider the cell as spiking
    #pkct_max: an optional parameter, max number of peaks to consider the cell as spiking
#outputs:
    #peak_data: an n-by-t 2D numpy array. It is same as the 'data' input except data for cells considered not spiking have been removed and replaced with NaN
    #peak_locations: a list of length n. Contains [np.nan] for cells not spiking. Contains the numpy array of indices where spiking happened for cells that had
    #                spiking events
    #peak_binary: a list of length n. Contains 1 for cells that are spiking, 0 for cells that are not spiking
    #peak_height: a list of length n. Contains 0 for cells not spiking. Contains the numpy array of prominences where spiking happened for cells that had
    #             spiking events
def select_spiking(data, prom, pkct_min=3, pkct_max=100):
    def count_peaks(x, p):
        peaks, properties = find_peaks(x, prominence = p)
        return peaks, properties

    peak_data=[]
    peak_locations = []

    peak_binary = []
    pk_height = []

    for i in range(data.shape[0]):
        pks, pp = count_peaks(data[i,:], prom)
        count = pks.shape[0]
        if (count > pkct_min and count < pkct_max):
            peak_data.append(data[i,:])
            peak_locations.append(pks)
            peak_binary.append(1)
            pk_height.append(pp['prominences'].mean())
        else:
            empty_ex = np.empty(data[i,:].shape)
            empty_ex[:]=np.nan
            peak_data.append(empty_ex)
            peak_locations.append(np.array([np.nan]))
            peak_binary.append(0)
            pk_height.append(0)

    return np.asarray(peak_data), peak_locations, peak_binary, pk_height

#######################################################################################################################################################
#function to generate spiking frequency of the spiking cells. It takes the average number of timepoints between each spiking event for the cells
#inputs:
    #j: an n-by-t 2D numpy array. It contains average calcium intensity for cells except data for cells considered not spiking have been
    #   removed and replaced with NaN. Use output from the select_spiking() function
    #threshold: integer value for the maximum interval between spiking events. Any interval higher than this is ignored
#outputs:
    #freq: an n-by-1 1D numpy array containing the average interval length between each spike for each cell. It contains np.nan for cells that do not have spiking
def get_cell_freq(j, threshold):

    freq = np.zeros(len(j))
    for i in range(len(j)):
        if np.any(j[i] == np.nan):
            freq[i] = 0
        else:
            dif=0
            ind=0
            for pk in range(j[i].shape[0]-1):
                if (j[i][pk+1] - j[i][pk]) <= threshold:
                    dif = dif + j[i][pk+1] - j[i][pk]
                    ind=ind+1
            try:
                freq[i]= dif/ind
            except:
                freq[i]=0
    freq[freq==0] = np.nan
    freq[freq > threshold] = np.nan
    return freq


In [ ]:
#image file name and the timepoints to take here.
imgset = tifffile.imread('Oua_50nM_Gray.tif')[0:240,:,:]

In [ ]:
data, mask, morph, size= extract_data_wloc(imgset, imgset[0,:,:])
data = poly_corr(data, 6)
peak_data, peak_locations, peak_binary, pk_height = select_spiking(data, 80)
pk_flt = []
for i in range(len(peak_locations)):
    if peak_binary[i]==1:
        pk_flt.append(peak_locations[i])

print("Fraction of cells deemed spiking")
print(len(pk_flt)/len(peak_locations))
x = get_cell_freq(pk_flt, 40)

## BMPS-style burst-duration quantification

This section incorporates the burst-duration logic from the `bMPS_analysis_tools` calcium-imaging notebook into this `Calcium_Frequency` workflow.

Conceptually, the bMPS notebook detects calcium peaks, computes peak prominence, then uses `scipy.signal.peak_widths(..., rel_height=1)` and treats the resulting full-prominence width (`res_full[0]`) as burst duration. Here, the same width is converted from frames to seconds using `FRAME_INTERVAL_S`.

Run this section **after** the existing cell that creates corrected `data`:

```python
data, mask, morph, size = extract_data_wloc(...)
data = poly_corr(data, 6)
```

Terminology: these outputs are calcium transient / burst-duration features, not direct action potentials.

In [ ]:
# BMPS-style burst-duration functions
# This cell is self-contained and only requires numpy, pandas, and scipy.

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_prominences, peak_widths


def _safe_mean(x):
    x = np.asarray(x, dtype=float)
    return np.nan if x.size == 0 else float(np.nanmean(x))


def _safe_median(x):
    x = np.asarray(x, dtype=float)
    return np.nan if x.size == 0 else float(np.nanmedian(x))


def _safe_std(x):
    x = np.asarray(x, dtype=float)
    return np.nan if x.size == 0 else float(np.nanstd(x, ddof=0))


def _contiguous_regions(mask):
    """Return start/end frame indices for True regions in a 1D Boolean mask."""
    mask = np.asarray(mask, dtype=bool)
    regions = []
    in_region = False
    start = None
    for idx, value in enumerate(mask):
        if value and not in_region:
            in_region = True
            start = idx
        elif not value and in_region:
            regions.append((start, idx - 1))
            in_region = False
    if in_region:
        regions.append((start, len(mask) - 1))
    return regions


def analyze_bmps_burst_duration(
    data,
    frame_interval_s,
    prominence=80,
    min_peak_count=3,
    max_peak_count=100,
    peak_width_rel_height=1.0,
    half_width_rel_height=0.5,
    wlen=None,
    min_peak_distance_frames=None,
    min_peak_width_frames=None,
    organoid_id="Organoid_01",
    condition="Condition_01",
    network_participation_threshold=0.20,
    min_network_burst_duration_s=0.0,
):
    """
    BMPS-style burst-duration analysis for the Calcium_Frequency workflow.

    Parameters
    ----------
    data : ndarray, shape (n_rois, n_frames)
        ROI x time fluorescence traces. This is designed to run after your existing
        extraction and polynomial correction steps:
            data, mask, morph, size = extract_data_wloc(...)
            data = poly_corr(data, 6)
    frame_interval_s : float
        Seconds per image frame. Use 5.0 if your acquisition interval is 5 seconds.
    prominence : float
        Prominence threshold passed to scipy.signal.find_peaks.
    min_peak_count : int
        Minimum number of peaks required for ROI to pass the original-style
        spiking/activity filter. Uses >= min_peak_count.
    max_peak_count : int
        Maximum number of peaks allowed for ROI to pass the activity filter.
    peak_width_rel_height : float
        rel_height for scipy.signal.peak_widths. The bMPS notebook used rel_height=1
        and called the resulting full-prominence width burst duration.
    half_width_rel_height : float
        rel_height for half-width calculation. Usually 0.5.
    wlen : int or None
        Window length used by scipy.signal.peak_prominences. None uses full trace.
    min_peak_distance_frames : int or None
        Optional minimum distance between detected peaks.
    min_peak_width_frames : int or None
        Optional minimum width for detected peaks.
    organoid_id, condition : str
        Metadata added to output tables.
    network_participation_threshold : float
        Fraction of ROIs that must be active at a frame to call a network burst.
    min_network_burst_duration_s : float
        Minimum network-burst duration in seconds.

    Returns
    -------
    dict with:
        event_table : one row per detected peak/calcium transient
        roi_summary : one row per ROI
        organoid_summary : one-row summary for this imaging file/organoid
        network_bursts : one row per network burst
        binary_activity : ROI x frame Boolean matrix using full-width event windows
        population_activity : fraction of ROIs active at each frame
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 2:
        raise ValueError("data must be a 2D array with shape (n_rois, n_frames)")
    if frame_interval_s <= 0:
        raise ValueError("frame_interval_s must be positive")

    n_rois, n_frames = data.shape
    recording_duration_s = n_frames * frame_interval_s
    recording_duration_min = recording_duration_s / 60.0 if recording_duration_s > 0 else np.nan

    event_rows = []
    roi_rows = []
    binary_activity = np.zeros((n_rois, n_frames), dtype=bool)

    find_peak_kwargs = {"prominence": prominence}
    if min_peak_distance_frames is not None:
        find_peak_kwargs["distance"] = int(min_peak_distance_frames)
    if min_peak_width_frames is not None:
        find_peak_kwargs["width"] = int(min_peak_width_frames)

    for roi in range(n_rois):
        trace = np.asarray(data[roi, :], dtype=float)
        valid = np.isfinite(trace)

        if valid.sum() < 3:
            peaks = np.array([], dtype=int)
            full_widths_frames = np.array([], dtype=float)
            half_widths_frames = np.array([], dtype=float)
            prominences = np.array([], dtype=float)
            left_ips_full = np.array([], dtype=float)
            right_ips_full = np.array([], dtype=float)
        else:
            clean_trace = trace.copy()
            if not valid.all():
                # Replace sparse NaNs by the median so scipy can still run.
                clean_trace[~valid] = np.nanmedian(clean_trace[valid])

            peaks, peak_props = find_peaks(clean_trace, **find_peak_kwargs)

            if len(peaks) > 0:
                prom_data = peak_prominences(clean_trace, peaks, wlen=wlen)
                prominences = prom_data[0]

                res_full = peak_widths(
                    clean_trace,
                    peaks,
                    rel_height=peak_width_rel_height,
                    prominence_data=prom_data,
                )
                full_widths_frames = res_full[0]
                left_ips_full = res_full[2]
                right_ips_full = res_full[3]

                res_half = peak_widths(
                    clean_trace,
                    peaks,
                    rel_height=half_width_rel_height,
                    prominence_data=prom_data,
                )
                half_widths_frames = res_half[0]
            else:
                full_widths_frames = np.array([], dtype=float)
                half_widths_frames = np.array([], dtype=float)
                prominences = np.array([], dtype=float)
                left_ips_full = np.array([], dtype=float)
                right_ips_full = np.array([], dtype=float)

        n_peaks = int(len(peaks))
        passes_peak_count_filter = (n_peaks >= min_peak_count) and (n_peaks <= max_peak_count)
        is_active = n_peaks > 0

        for event_id, peak_frame in enumerate(peaks):
            onset_frame_float = float(left_ips_full[event_id])
            end_frame_float = float(right_ips_full[event_id])
            onset_frame = max(0, int(np.floor(onset_frame_float)))
            end_frame = min(n_frames - 1, int(np.ceil(end_frame_float)))
            if end_frame >= onset_frame:
                binary_activity[roi, onset_frame:end_frame + 1] = True

            event_rows.append({
                "organoid_id": organoid_id,
                "condition": condition,
                "roi": roi,
                "event_id": event_id,
                "peak_frame": int(peak_frame),
                "peak_time_s": float(peak_frame * frame_interval_s),
                "peak_prominence": float(prominences[event_id]),
                "burst_duration_frames_bmps_full_width": float(full_widths_frames[event_id]),
                "burst_duration_s_bmps_full_width": float(full_widths_frames[event_id] * frame_interval_s),
                "half_width_frames": float(half_widths_frames[event_id]),
                "half_width_s": float(half_widths_frames[event_id] * frame_interval_s),
                "onset_frame_interp": onset_frame_float,
                "end_frame_interp": end_frame_float,
                "onset_time_s_interp": onset_frame_float * frame_interval_s,
                "end_time_s_interp": end_frame_float * frame_interval_s,
            })

        roi_rows.append({
            "organoid_id": organoid_id,
            "condition": condition,
            "roi": roi,
            "is_active": bool(is_active),
            "passes_peak_count_filter": bool(passes_peak_count_filter),
            "n_peaks": n_peaks,
            "event_frequency_per_min": n_peaks / recording_duration_min if recording_duration_min and recording_duration_min > 0 else np.nan,
            "mean_burst_duration_s_bmps_full_width": _safe_mean(full_widths_frames * frame_interval_s),
            "median_burst_duration_s_bmps_full_width": _safe_median(full_widths_frames * frame_interval_s),
            "std_burst_duration_s_bmps_full_width": _safe_std(full_widths_frames * frame_interval_s),
            "mean_half_width_s": _safe_mean(half_widths_frames * frame_interval_s),
            "median_half_width_s": _safe_median(half_widths_frames * frame_interval_s),
            "std_half_width_s": _safe_std(half_widths_frames * frame_interval_s),
            "mean_peak_prominence": _safe_mean(prominences),
            "total_normalized_activation_time_bmps": float(np.nansum(full_widths_frames) / n_frames) if n_frames > 0 else np.nan,
        })

    event_table = pd.DataFrame(event_rows)
    roi_summary = pd.DataFrame(roi_rows)

    # Network-level bursts from the ROI activity windows.
    population_activity = binary_activity.sum(axis=0) / n_rois if n_rois > 0 else np.array([])
    burst_mask = population_activity >= network_participation_threshold
    min_network_burst_frames = int(np.ceil(min_network_burst_duration_s / frame_interval_s)) if frame_interval_s > 0 else 0
    network_rows = []
    for network_event_id, (start, end) in enumerate(_contiguous_regions(burst_mask)):
        duration_frames = end - start + 1
        if duration_frames < max(1, min_network_burst_frames):
            continue
        network_rows.append({
            "organoid_id": organoid_id,
            "condition": condition,
            "network_event_id": network_event_id,
            "start_frame": int(start),
            "end_frame": int(end),
            "start_time_s": float(start * frame_interval_s),
            "end_time_s": float(end * frame_interval_s),
            "network_burst_duration_s": float(duration_frames * frame_interval_s),
            "max_population_participation": float(np.nanmax(population_activity[start:end + 1])),
            "mean_population_participation": float(np.nanmean(population_activity[start:end + 1])),
        })
    network_bursts = pd.DataFrame(network_rows)

    active_indices = roi_summary.index[roi_summary["is_active"]].to_numpy()
    if len(active_indices) >= 2:
        corr_matrix = np.corrcoef(data[active_indices, :])
        upper = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
        synchrony_index = float(np.nanmean(upper))
    else:
        synchrony_index = np.nan

    organoid_summary = pd.DataFrame([{
        "organoid_id": organoid_id,
        "condition": condition,
        "n_rois": n_rois,
        "n_frames": n_frames,
        "frame_interval_s": frame_interval_s,
        "recording_duration_s": recording_duration_s,
        "active_cell_fraction": float(roi_summary["is_active"].mean()) if n_rois > 0 else np.nan,
        "passes_peak_count_filter_fraction": float(roi_summary["passes_peak_count_filter"].mean()) if n_rois > 0 else np.nan,
        "median_event_frequency_per_min": float(roi_summary["event_frequency_per_min"].median()),
        "mean_burst_duration_s_bmps_full_width": float(roi_summary["mean_burst_duration_s_bmps_full_width"].mean()),
        "median_burst_duration_s_bmps_full_width": float(roi_summary["median_burst_duration_s_bmps_full_width"].median()),
        "median_half_width_s": float(roi_summary["median_half_width_s"].median()),
        "network_burst_count": int(len(network_bursts)),
        "network_burst_frequency_per_min": float(len(network_bursts) / recording_duration_min) if recording_duration_min and recording_duration_min > 0 else np.nan,
        "mean_network_burst_duration_s": float(network_bursts["network_burst_duration_s"].mean()) if len(network_bursts) else np.nan,
        "median_network_burst_duration_s": float(network_bursts["network_burst_duration_s"].median()) if len(network_bursts) else np.nan,
        "synchrony_index_raw_trace_corr_active_rois": synchrony_index,
    }])

    return {
        "event_table": event_table,
        "roi_summary": roi_summary,
        "organoid_summary": organoid_summary,
        "network_bursts": network_bursts,
        "binary_activity": binary_activity,
        "population_activity": population_activity,
    }


In [ ]:
# Run BMPS-style burst-duration analysis on the existing Calcium_Frequency `data` matrix.
# `data` should be ROI x frame after your original extraction and polynomial correction.

# ---- USER SETTINGS ----
FRAME_INTERVAL_S = 5.0      # seconds per frame. Change this if your acquisition interval is different.
PROMINENCE = 80             # matches your original select_spiking(data, 80) setting.
MIN_PEAK_COUNT = 3          # ROI is activity-filter-positive if it has at least this many peaks.
MAX_PEAK_COUNT = 100
ORGANOID_ID = "Oua_50nM_Organoid_01"
CONDITION = "NorHA_or_Matrigel"
NETWORK_PARTICIPATION_THRESHOLD = 0.20  # network burst when >=20% ROIs are active.
MIN_NETWORK_BURST_DURATION_S = 0.0

# ---- ANALYSIS ----
burst_results = analyze_bmps_burst_duration(
    data=data,
    frame_interval_s=FRAME_INTERVAL_S,
    prominence=PROMINENCE,
    min_peak_count=MIN_PEAK_COUNT,
    max_peak_count=MAX_PEAK_COUNT,
    peak_width_rel_height=1.0,       # bMPS-style full-prominence burst duration
    half_width_rel_height=0.5,       # useful companion metric
    organoid_id=ORGANOID_ID,
    condition=CONDITION,
    network_participation_threshold=NETWORK_PARTICIPATION_THRESHOLD,
    min_network_burst_duration_s=MIN_NETWORK_BURST_DURATION_S,
)

event_table = burst_results["event_table"]
roi_summary = burst_results["roi_summary"]
organoid_summary = burst_results["organoid_summary"]
network_bursts = burst_results["network_bursts"]
population_activity = burst_results["population_activity"]
binary_activity = burst_results["binary_activity"]

# ---- SAVE OUTPUTS ----
event_table.to_csv(f"{ORGANOID_ID}_{CONDITION}_bmps_event_table.csv", index=False)
roi_summary.to_csv(f"{ORGANOID_ID}_{CONDITION}_bmps_roi_summary.csv", index=False)
organoid_summary.to_csv(f"{ORGANOID_ID}_{CONDITION}_bmps_organoid_summary.csv", index=False)
network_bursts.to_csv(f"{ORGANOID_ID}_{CONDITION}_network_bursts.csv", index=False)

print("Saved output CSV files.")
print("Event table shape:", event_table.shape)
print("ROI summary shape:", roi_summary.shape)
print("Network bursts shape:", network_bursts.shape)

organoid_summary


In [ ]:
# Quick plots for QC and interpretation.
# 1) Distribution of BMPS-style burst duration per calcium event.
# 2) Population activity over time with network-burst threshold.

import matplotlib.pyplot as plt
import numpy as np

if len(event_table) > 0:
    plt.figure(figsize=(8, 5))
    plt.hist(event_table["burst_duration_s_bmps_full_width"].dropna(), bins=30)
    plt.xlabel("BMPS-style burst duration, full-prominence width (s)")
    plt.ylabel("Number of calcium events")
    plt.title(f"{ORGANOID_ID} | {CONDITION}: burst-duration distribution")
    plt.show()
else:
    print("No calcium events detected. Consider lowering PROMINENCE or checking trace quality.")

if len(population_activity) > 0:
    time_s = np.arange(len(population_activity)) * FRAME_INTERVAL_S
    plt.figure(figsize=(12, 4))
    plt.plot(time_s, population_activity)
    plt.axhline(NETWORK_PARTICIPATION_THRESHOLD, linestyle="--")
    plt.xlabel("Time (s)")
    plt.ylabel("Fraction of ROIs active")
    plt.title("Population activity and network-burst threshold")
    plt.show()

network_bursts.head()


In [ ]:
# Optional: inspect one ROI trace with detected peaks and BMPS-style full-width burst windows.
# Change ROI_TO_PLOT to any ROI number present in event_table.

ROI_TO_PLOT = 0

trace = data[ROI_TO_PLOT, :]
roi_events = event_table[event_table["roi"] == ROI_TO_PLOT].copy()
time_s = np.arange(data.shape[1]) * FRAME_INTERVAL_S

plt.figure(figsize=(12, 4))
plt.plot(time_s, trace)

if len(roi_events) > 0:
    plt.scatter(roi_events["peak_time_s"], trace[roi_events["peak_frame"].astype(int)], marker="x")
    for _, row in roi_events.iterrows():
        plt.axvspan(row["onset_time_s_interp"], row["end_time_s_interp"], alpha=0.15)

plt.xlabel("Time (s)")
plt.ylabel("Corrected fluorescence, arbitrary units")
plt.title(f"ROI {ROI_TO_PLOT}: detected calcium events and BMPS-style burst windows")
plt.show()

roi_summary[roi_summary["roi"] == ROI_TO_PLOT]


In [ ]:
plt.figure(figsize = (12,5));
plt.hist(1/(x*5)*1000, bins=np.arange(1,25,2));
plt.xticks(fontsize=20);
plt.yticks(fontsize=20);
print("The average frequency is")
print(np.nanmean(1/(x*5)*1000))


In [ ]:
#output the peak info as a csv file
np.savetxt('n4.csv', np.append(np.asarray([peak_binary]).T, data, axis=1), delimiter=',')
file = open('n4_peaks.txt', 'w')

for peak in peak_locations:
    file.write(' '.join([str(i) for i in peak]) + '\n')
file.close()

In [ ]:
#snippet to remove nan data (nonspiking cells)
peak_data = peak_data[~np.isnan(peak_data).any(axis=1)]
y=[i for i in peak_locations if i[0]!='nan']

#get the peak timepoint locations for each cell
pk_flt = []
for i in range(len(peak_locations)):
    if peak_binary[i]==1:
        pk_flt.append(peak_locations[i])